# AI Search Configuraiton & Setup

### Pulling data into an index from blob storage requires the following to be setup:

- Data Source
- Index
- Skillset
- Indexer

The pull model uses indexers connecting to a supported data source, automatically uploading the data into your index.

Note to leverage this notebook requires:

- Install .NET 8
- Launch VS Code and install "Polyglot" extention

## AI Search Index Creation


In [41]:
#r "nuget: dotenv.net, 3.0.0"
#r "nuget: Newtonsoft.Json"

Installed Packages dotenv.net, 3.0.0 Newtonsoft.Json, 13.0.3

In [53]:
using System;  
using System.IO;  
using System.Net.Http;  
using System.Text;  
using System.Threading.Tasks;  
using Newtonsoft.Json; 
  
public static void Load(string filePath)  
{  
    if (!File.Exists(filePath))  
        return;  
  
    foreach (var line in File.ReadAllLines(filePath))  
    {  
        // Split the line into exactly two parts  
        var parts = line.Split(new char[] {'='}, 2);  
  
        if (parts.Length != 2)  
            continue;  // Skip lines that don't properly split into two parts  
  
        // Trim parts to remove any accidental whitespace  
        string key = parts[0].Trim();  
        string value = parts[1].Trim();  
  
        Environment.SetEnvironmentVariable(key, value);  
  
        // Optional: Add a debug line to confirm what's being set  
        //Console.WriteLine($"Setting {key} to {value}");  
    }  
}  

public static void CheckEmpty(string variableName)  
{  
    // Retrieve the value of the environment variable  
    string value = Environment.GetEnvironmentVariable(variableName);  
  
    // Check if the value is null or empty  
    if (string.IsNullOrEmpty(value))  
    {  
        Console.WriteLine($"{variableName} is empty or not set.");  
    }  
    // else  
    // {  
    //     Console.WriteLine($"{variableName} is set to: {value}");  
    // }  
}  

In [54]:
Load(".env");

// Check for empty environment variables  
CheckEmpty("AZURE_SEARCH_SERVICE_NAME");  
CheckEmpty("AZURE_SEARCH_INDEX");  
CheckEmpty("AZURE_SEARCH_API_KEY");  
CheckEmpty("BLOB_CONNECTION_STRING");  
CheckEmpty("BLOB_CONTAINER_NAME");  
CheckEmpty("AZURE_OPENAI_ENDPOINT");  
CheckEmpty("AZURE_OPENAI_EMBEDDING_MODEL_NAME");  
CheckEmpty("AZURE_OPENAI_EMBEDDING_DIMENSIONS");  
CheckEmpty("AZURE_AI_SERVICES_ENDPOINT"); 

string azureSearchServiceName = Environment.GetEnvironmentVariable("AZURE_SEARCH_SERVICE_NAME");  
string azureSearchIndex = Environment.GetEnvironmentVariable("AZURE_SEARCH_INDEX");  
string azureSearchIndexLayout = Environment.GetEnvironmentVariable("AZURE_SEARCH_INDEX_LAYOUT");  
string azureSearchIndexOcr = Environment.GetEnvironmentVariable("AZURE_SEARCH_INDEX_OCR");  
string azureSearchApiKey = Environment.GetEnvironmentVariable("AZURE_SEARCH_API_KEY");  
string blobConnectionString = Environment.GetEnvironmentVariable("BLOB_CONNECTION_STRING")?.Trim('"'); 
string blobContainerName = Environment.GetEnvironmentVariable("BLOB_CONTAINER_NAME");  
string azureOpenAiEndpoint = Environment.GetEnvironmentVariable("AZURE_OPENAI_ENDPOINT");  
string azureOpenAiEmbeddingModelName = Environment.GetEnvironmentVariable("AZURE_OPENAI_EMBEDDING_MODEL_NAME");  
string azureOpenAiEmbeddingDimensions = Environment.GetEnvironmentVariable("AZURE_OPENAI_EMBEDDING_DIMENSIONS");  
string azureAiServicesEndpoint = Environment.GetEnvironmentVariable("AZURE_AI_SERVICES_ENDPOINT");  
  

Console.WriteLine("Variables have been set");

Variables have been set


## Create Data Source

https://learn.microsoft.com/en-us/azure/search/search-howto-indexing-azure-blob-storage

In [55]:
public static void CreateDataSource(string serviceName, string indexName, string searchApiKey, string storageConnectionString, string storageContainer)  
{  
    Console.WriteLine("AZURE_SEARCH_INDEX = {0}", indexName);  
    Console.WriteLine("index_name = {0}", indexName);  

    string endpoint = $"https://{serviceName}.search.windows.net/";  
    string url = $"{endpoint}/datasources/{indexName}-datasource?api-version=2024-11-01-preview";  

    var payload = $@"  
    {{  
        ""name"": ""{indexName}-datasource"",  
        ""description"": null,  
        ""type"": ""azureblob"",  
        ""subtype"": null,  
        ""credentials"": {{  
            ""connectionString"": ""{storageConnectionString}""  
        }},  
        ""container"": {{  
            ""name"": ""{storageContainer}"",  
            ""query"": null  
        }},  
        ""dataChangeDetectionPolicy"": null,  
        ""dataDeletionDetectionPolicy"": {{  
            ""@odata.type"": ""#Microsoft.Azure.Search.NativeBlobSoftDeleteDeletionDetectionPolicy""  
        }},  
        ""encryptionKey"": null,  
        ""identity"": null  
    }}";  

    using (var client = new HttpClient())  
    {  
        client.DefaultRequestHeaders.Add("api-key", searchApiKey);  
        var content = new StringContent(payload, Encoding.UTF8, "application/json");  

        HttpResponseMessage response = client.PutAsync(url, content).GetAwaiter().GetResult();  
        Console.WriteLine("Response: {0}", response); // Print the response
        Console.WriteLine("HTTP Status Code: {0}", response.StatusCode); // Print the status code  

        if (!response.IsSuccessStatusCode)  
        {  
            Console.WriteLine("Error creating data source: {0}", response.Content.ReadAsStringAsync().GetAwaiter().GetResult());  
        }  
    }  
}  

In [56]:
CreateDataSource(azureSearchServiceName, azureSearchIndex, azureSearchApiKey, blobConnectionString, blobContainerName); 

AZURE_SEARCH_INDEX = vector-manual24
index_name = vector-manual24
Response: StatusCode: 201, ReasonPhrase: 'Created', Version: 1.1, Content: System.Net.Http.HttpConnectionResponseContent, Headers:
{
  Transfer-Encoding: chunked
  ETag: "0x8DD766171CEC675"
  Location: https://mmchrdemo03search.search.windows.net:443/datasources('vector-manual24-datasource')?api-version=2024-11-01-preview
  Server: Microsoft-IIS/10.0
  Strict-Transport-Security: max-age=2592000
  Strict-Transport-Security: max-age=15724800; includeSubDomains
  Preference-Applied: odata.include-annotations="*"
  OData-Version: 4.0
  request-id: 74e0a24f-0632-4a4c-a8a7-5c3482c9b288
  elapsed-time: 83
  Date: Tue, 08 Apr 2025 05:51:44 GMT
  Content-Type: application/json; odata.metadata=minimal; odata.streaming=true; charset=utf-8
}
HTTP Status Code: Created


## Create Search 

https://learn.microsoft.com/en-us/azure/search/search-get-started-portal

https://learn.microsoft.com/en-us/azure/search/vector-search-ranking

For HNSW, try different levels of efConstruction to change the internal composition of the proximity graph. The default is 400. The range is 100 to 1,000.

In [57]:
public static void CreateIndexSemantic(string serviceName, string index, string azureSearchApiKey, string openaiApiBase, string textEmbbedingModel, int textEmbbedingModelDimensions)  
    {  
        string endpoint = $"https://{serviceName}.search.windows.net/";  
        string url = $"{endpoint}/indexes/{index}/?api-version=2024-11-01-preview";  
        Console.WriteLine(url);  
  
        var payload = $@"  
        {{  
            ""name"": ""{index}"",  
            ""fields"": [  
                {{""name"": ""chunk_id"", ""type"": ""Edm.String"", ""searchable"": true, ""filterable"": false, ""retrievable"": true, ""stored"": true, ""sortable"": true, ""facetable"": false, ""key"": true, ""analyzer"": ""keyword""}},  
                {{""name"": ""parent_id"", ""type"": ""Edm.String"", ""searchable"": false, ""filterable"": true, ""retrievable"": true, ""stored"": true, ""sortable"": false, ""facetable"": false, ""key"": false}},  
                {{""name"": ""chunk"", ""type"": ""Edm.String"", ""searchable"": true, ""filterable"": false, ""retrievable"": true, ""stored"": true, ""sortable"": false, ""facetable"": false, ""key"": false}},  
                {{""name"": ""title"", ""type"": ""Edm.String"", ""searchable"": true, ""filterable"": false, ""retrievable"": true, ""stored"": true, ""sortable"": false, ""facetable"": false, ""key"": false}},  
                {{""name"": ""header_1"", ""type"": ""Edm.String"", ""searchable"": true, ""filterable"": false, ""retrievable"": true, ""stored"": true, ""sortable"": false, ""facetable"": false, ""key"": false}},  
                {{""name"": ""header_2"", ""type"": ""Edm.String"", ""searchable"": true, ""filterable"": false, ""retrievable"": true, ""stored"": true, ""sortable"": false, ""facetable"": false, ""key"": false}},  
                {{""name"": ""header_3"", ""type"": ""Edm.String"", ""searchable"": true, ""filterable"": false, ""retrievable"": true, ""stored"": true, ""sortable"": false, ""facetable"": false, ""key"": false}},  
                {{""name"": ""loads"", ""type"": ""Collection(Edm.String)"", ""searchable"": true, ""filterable"": true, ""retrievable"": true, ""stored"": true, ""sortable"": false, ""facetable"": false, ""key"": false}},  
                {{""name"": ""text_vector"", ""type"": ""Collection(Edm.Single)"", ""searchable"": true, ""filterable"": false, ""retrievable"": true, ""stored"": true, ""sortable"": false, ""facetable"": false, ""key"": false, ""dimensions"": {textEmbbedingModelDimensions}, ""vectorSearchProfile"": ""{index}-azureOpenAi-text-profile""}}  
            ],  
            ""scoringProfiles"": [],  
            ""suggesters"": [],  
            ""analyzers"": [],  
            ""normalizers"": [],  
            ""tokenizers"": [],  
            ""tokenFilters"": [],  
            ""charFilters"": [],  
            ""similarity"": {{  
                ""@odata.type"": ""#Microsoft.Azure.Search.BM25Similarity""  
            }},  
            ""semantic"": {{  
                ""configurations"": [  
                    {{  
                        ""name"": ""{index}-semantic-configuration"",  
                        ""prioritizedFields"": {{  
                            ""titleField"": {{""fieldName"": ""title""}},  
                            ""prioritizedContentFields"": [{{""fieldName"": ""chunk""}}]  
                        }}  
                    }}  
                ]  
            }},  
            ""vectorSearch"": {{  
                ""algorithms"": [  
                    {{  
                        ""name"": ""{index}-algorithm"",  
                        ""kind"": ""hnsw"",  
                        ""hnswParameters"": {{  
                            ""metric"": ""cosine"",  
                            ""m"": 4,  
                            ""efConstruction"": 400,  
                            ""efSearch"": 500  
                        }}  
                    }}  
                ],  
                ""profiles"": [  
                    {{  
                        ""name"": ""{index}-azureOpenAi-text-profile"",  
                        ""algorithm"": ""{index}-algorithm"",  
                        ""vectorizer"": ""{index}-azureOpenAi-text-vectorizer""  
                    }}  
                ],  
                ""vectorizers"": [  
                    {{  
                        ""name"": ""{index}-azureOpenAi-text-vectorizer"",  
                        ""kind"": ""azureOpenAI"",  
                        ""azureOpenAIParameters"": {{  
                            ""resourceUri"": ""{openaiApiBase}"",  
                            ""deploymentId"": ""{textEmbbedingModel}"",  
                            ""modelName"": ""{textEmbbedingModel}""  
                        }}  
                    }}  
                ],  
                ""compressions"": []  
            }}  
        }}";  
  
        //Console.WriteLine(payload);
        using (var client = new HttpClient())  
        {  
            client.DefaultRequestHeaders.Add("api-key", azureSearchApiKey);  
            var content = new StringContent(payload, Encoding.UTF8, "application/json");  
    
            HttpResponseMessage response = client.PutAsync(url, content).GetAwaiter().GetResult();  
            Console.WriteLine("Response: {0}", response); // Print the response
            Console.WriteLine("HTTP Status Code: {0}", response.StatusCode); // Print the status code  
    
            if (!response.IsSuccessStatusCode)  
            {  
                Console.WriteLine("Error creating data source: {0}", response.Content.ReadAsStringAsync().GetAwaiter().GetResult());  
            }  
        }  
    }  

In [58]:
CreateIndexSemantic(azureSearchServiceName, azureSearchIndex, azureSearchApiKey, azureOpenAiEndpoint, azureOpenAiEmbeddingModelName, int.Parse(azureOpenAiEmbeddingDimensions));

https://mmchrdemo03search.search.windows.net//indexes/vector-manual24/?api-version=2024-11-01-preview
Response: StatusCode: 201, ReasonPhrase: 'Created', Version: 1.1, Content: System.Net.Http.HttpConnectionResponseContent, Headers:
{
  Transfer-Encoding: chunked
  ETag: "0x8DD766172CCF00A"
  Location: https://mmchrdemo03search.search.windows.net:443/indexes('vector-manual24')?api-version=2024-11-01-preview
  Server: Microsoft-IIS/10.0
  Strict-Transport-Security: max-age=2592000
  Strict-Transport-Security: max-age=15724800; includeSubDomains
  Preference-Applied: odata.include-annotations="*"
  OData-Version: 4.0
  request-id: 15789ba7-1047-43c1-8554-781eee6c1c1f
  elapsed-time: 339
  Date: Tue, 08 Apr 2025 05:51:46 GMT
  Content-Type: application/json; odata.metadata=minimal; odata.streaming=true; charset=utf-8
}
HTTP Status Code: Created


## Create a Skillset
- Layout Skill
    -  https://learn.microsoft.com/en-us/azure/search/cognitive-search-skill-document-intelligence-layout
- WebApi Skill
- Split Skill
    - https://learn.microsoft.com/en-us/azure/search/cognitive-search-skill-textsplit
- AzureOpenAIEmbeddingSkill
    - https://learn.microsoft.com/en-us/azure/search/cognitive-search-skill-azure-openai-embedding

In [59]:
public static void CreateSkillset(string serviceName, string searchApiKey, string index, string openaiApiBase, string textEmbeddingModel, string azureAiServicesEndpoint, string textEmbeddingModelDimensions)  
{  
    string endpoint = $"https://{serviceName}.search.windows.net/";  
    string resourceUri = openaiApiBase;  
    string deploymentId = textEmbeddingModel;  
    string modelName = textEmbeddingModel;  
    string dimensions = textEmbeddingModelDimensions;  
    string embeddingFunctionAppUriAndKey = Environment.GetEnvironmentVariable("AZURE_FUNCTION_APP_URL");  
    string azureAiServicesKey = Environment.GetEnvironmentVariable("AZURE_AI_SERVICES_KEY");  
  
    string url = $"{endpoint}/skillsets/{index}-skillset?api-version=2024-11-01-preview";  
    Console.WriteLine(url);  
  
    var payload = $@"  
    {{  
        ""name"": ""{index}-skillset"",  
        ""description"": ""Skillset to chunk documents and generate embeddings"",  
        ""skills"": [  
            {{  
                ""@odata.type"": ""#Microsoft.Skills.Util.DocumentIntelligenceLayoutSkill"",  
                ""name"": ""#1"",  
                ""context"": ""/document"",  
                ""outputMode"": ""oneToMany"",  
                ""markdownHeaderDepth"": ""h3"",  
                ""inputs"": [  
                    {{  
                        ""name"": ""file_data"",  
                        ""source"": ""/document/file_data""  
                    }}  
                ],  
                ""outputs"": [  
                    {{  
                        ""name"": ""markdown_document"",  
                        ""targetName"": ""markdownDocument""  
                    }}  
                ]  
            }},  
            {{  
                ""@odata.type"": ""#Microsoft.Skills.Custom.WebApiSkill"",  
                ""uri"": ""{embeddingFunctionAppUriAndKey}"",  
                ""httpMethod"": ""POST"",  
                ""timeout"": ""PT230S"",  
                ""batchSize"": 1,  
                ""degreeOfParallelism"": 1,  
                ""name"": ""LoadFinder"",  
                ""context"": ""/document"",  
                ""inputs"": [  
                    {{  
                        ""name"": ""text"",  
                        ""source"": ""/document/markdownDocument""  
                    }}  
                ],  
                ""outputs"": [  
                    {{  
                        ""name"": ""loads"",  
                        ""targetName"": ""loads""  
                    }}  
                ]  
            }},  
            {{  
                ""@odata.type"": ""#Microsoft.Skills.Text.SplitSkill"",  
                ""name"": ""#2"",  
                ""description"": ""Split skill to chunk documents"",  
                ""context"": ""/document/markdownDocument/*"",  
                ""defaultLanguageCode"": ""en"",  
                ""textSplitMode"": ""pages"",  
                ""maximumPageLength"": 2000,  
                ""pageOverlapLength"": 500,  
                ""maximumPagesToTake"": 0,  
                ""unit"": ""characters"",  
                ""inputs"": [  
                    {{  
                        ""name"": ""text"",  
                        ""source"": ""/document/markdownDocument/*/content""  
                    }}  
                ],  
                ""outputs"": [  
                    {{  
                        ""name"": ""textItems"",  
                        ""targetName"": ""pages""  
                    }}  
                ]  
            }},  
            {{  
                ""@odata.type"": ""#Microsoft.Skills.Text.AzureOpenAIEmbeddingSkill"",  
                ""name"": ""#3"",  
                ""context"": ""/document/markdownDocument/*/pages/*"",  
                ""resourceUri"": ""{resourceUri}"",  
                ""deploymentId"": ""{deploymentId}"",  
                ""dimensions"": ""{dimensions}"",  
                ""modelName"": ""{modelName}"",  
                ""inputs"": [  
                    {{  
                        ""name"": ""text"",  
                        ""source"": ""/document/markdownDocument/*/pages/*""  
                    }}  
                ],  
                ""outputs"": [  
                    {{  
                        ""name"": ""embedding"",  
                        ""targetName"": ""text_vector""  
                    }}  
                ]  
            }}  
        ],  
        ""cognitiveServices"": {{  
            ""@odata.type"": ""#Microsoft.Azure.Search.AIServicesByKey"",  
            ""subdomainUrl"": ""{azureAiServicesEndpoint}"",  
            ""key"": ""{azureAiServicesKey}""  
        }},  
        ""indexProjections"": {{  
            ""selectors"": [  
                {{  
                    ""targetIndexName"": ""{index}"",  
                    ""parentKeyFieldName"": ""parent_id"",  
                    ""sourceContext"": ""/document/markdownDocument/*/pages/*"",  
                    ""mappings"": [  
                        {{  
                            ""name"": ""text_vector"",  
                            ""source"": ""/document/markdownDocument/*/pages/*/text_vector""  
                        }},  
                        {{  
                            ""name"": ""chunk"",  
                            ""source"": ""/document/markdownDocument/*/pages/*""  
                        }},  
                        {{  
                            ""name"": ""title"",  
                            ""source"": ""/document/title""  
                        }},  
                        {{  
                            ""name"": ""loads"",  
                            ""source"": ""/document/loads""  
                        }},  
                        {{  
                            ""name"": ""header_1"",  
                            ""source"": ""/document/markdownDocument/*/sections/h1""  
                        }},  
                        {{  
                            ""name"": ""header_2"",  
                            ""source"": ""/document/markdownDocument/*/sections/h2""  
                        }},  
                        {{  
                            ""name"": ""header_3"",  
                            ""source"": ""/document/markdownDocument/*/sections/h3""  
                        }}  
                    ]  
                }}  
            ],  
            ""parameters"": {{  
                ""projectionMode"": ""skipIndexingParentDocuments""  
            }}  
        }}  
    }}";  
  
    using (var client = new HttpClient())  
    {  
        client.DefaultRequestHeaders.Add("api-key", searchApiKey);  
        var content = new StringContent(payload, Encoding.UTF8, "application/json");  
  
        HttpResponseMessage response = client.PutAsync(url, content).GetAwaiter().GetResult();  
        Console.WriteLine("Response: {0}", response);  
        Console.WriteLine("HTTP Status Code: {0}", response.StatusCode);  
  
        if (!response.IsSuccessStatusCode)  
        {  
            Console.WriteLine("Error creating skillset: {0}", response.Content.ReadAsStringAsync().GetAwaiter().GetResult());  
        }  
    }  
}  

In [60]:
CreateSkillset(azureSearchServiceName, azureSearchApiKey, azureSearchIndex, azureOpenAiEndpoint, azureOpenAiEmbeddingModelName, azureAiServicesEndpoint, azureOpenAiEmbeddingDimensions);

https://mmchrdemo03search.search.windows.net//skillsets/vector-manual24-skillset?api-version=2024-11-01-preview
Response: StatusCode: 201, ReasonPhrase: 'Created', Version: 1.1, Content: System.Net.Http.HttpConnectionResponseContent, Headers:
{
  Transfer-Encoding: chunked
  ETag: "0x8DD766173926F14"
  Location: https://mmchrdemo03search.search.windows.net:443/skillsets('vector-manual24-skillset')?api-version=2024-11-01-preview
  Server: Microsoft-IIS/10.0
  Strict-Transport-Security: max-age=2592000
  Strict-Transport-Security: max-age=15724800; includeSubDomains
  Preference-Applied: odata.include-annotations="*"
  OData-Version: 4.0
  request-id: c5e5f2b6-4722-41f2-8fe7-3922705109f6
  elapsed-time: 85
  Date: Tue, 08 Apr 2025 05:51:47 GMT
  Content-Type: application/json; odata.metadata=minimal; odata.streaming=true; charset=utf-8
}
HTTP Status Code: Created


In [61]:
public static void CreateIndexer(string serviceName, string index, string searchKey)  
{  
    string endpoint = $"https://{serviceName}.search.windows.net/";  
    string url = $"{endpoint}/indexers/{index}-indexer/?api-version=2024-11-01-preview";  
    Console.WriteLine(url);  
  
    var payload = $@"  
    {{  
        ""name"": ""{index}-indexer"",  
        ""description"": null,  
        ""dataSourceName"": ""{index}-datasource"",  
        ""skillsetName"": ""{index}-skillset"",  
        ""targetIndexName"": ""{index}"",  
        ""disabled"": null,  
        ""schedule"": null,  
        ""parameters"": {{  
            ""batchSize"": null,  
            ""maxFailedItems"": null,  
            ""maxFailedItemsPerBatch"": null,  
            ""base64EncodeKeys"": null,  
            ""configuration"": {{  
                ""dataToExtract"": ""contentAndMetadata"",  
                ""parsingMode"": ""default"",  
                ""allowSkillsetToReadFileData"": true  
            }}  
        }},  
        ""fieldMappings"": [  
            {{  
                ""sourceFieldName"": ""metadata_storage_name"",  
                ""targetFieldName"": ""title"",  
                ""mappingFunction"": null  
            }}  
        ],  
        ""outputFieldMappings"": [],  
        ""cache"": null,  
        ""encryptionKey"": null  
    }}";  
  
    using (var client = new HttpClient())  
    {  
        client.DefaultRequestHeaders.Add("api-key", searchKey);  
        var content = new StringContent(payload, Encoding.UTF8, "application/json");  
  
        HttpResponseMessage response = client.PutAsync(url, content).GetAwaiter().GetResult();  
        Console.WriteLine("Response: {0}", response);  
        Console.WriteLine("HTTP Status Code: {0}", response.StatusCode);  
  
        if (!response.IsSuccessStatusCode)  
        {  
            Console.WriteLine("Error creating skillset: {0}", response.Content.ReadAsStringAsync().GetAwaiter().GetResult());  
        }  
    } 
}  

In [62]:
CreateIndexer(azureSearchServiceName, azureSearchIndex, azureSearchApiKey);

https://mmchrdemo03search.search.windows.net//indexers/vector-manual24-indexer/?api-version=2024-11-01-preview
Response: StatusCode: 201, ReasonPhrase: 'Created', Version: 1.1, Content: System.Net.Http.HttpConnectionResponseContent, Headers:
{
  Transfer-Encoding: chunked
  ETag: "0x8DD76617A89377F"
  Location: https://mmchrdemo03search.search.windows.net:443/indexers('vector-manual24-indexer')?api-version=2024-11-01-preview
  Server: Microsoft-IIS/10.0
  Strict-Transport-Security: max-age=2592000
  Strict-Transport-Security: max-age=15724800; includeSubDomains
  Preference-Applied: odata.include-annotations="*"
  OData-Version: 4.0
  request-id: 3a421fc3-6a22-4ca7-84b5-0777fa50bdef
  elapsed-time: 10917
  Date: Tue, 08 Apr 2025 05:51:59 GMT
  Content-Type: application/json; odata.metadata=minimal; odata.streaming=true; charset=utf-8
}
HTTP Status Code: Created


## Call Azure OpenAI with a DataSource

In [63]:
using System;
using Azure.AI.OpenAI;
using System.ClientModel;
using Azure.AI.OpenAI.Chat;
using OpenAI.Chat;
using static System.Environment;

string azureOpenAIEndpoint = GetEnvironmentVariable("AZURE_OPENAI_ENDPOINT");
string azureOpenAIKey = GetEnvironmentVariable("AZURE_OPENAI_API_KEY");
string deploymentName = GetEnvironmentVariable("AZURE_OPENAI_DEPLOYMENT_NAME");
string searchEndpoint = GetEnvironmentVariable("AZURE_AI_SEARCH_ENDPOINT");
string searchKey = GetEnvironmentVariable("AZURE_AI_SEARCH_API_KEY");
string searchIndex = GetEnvironmentVariable("AZURE_AI_SEARCH_INDEX");

AzureOpenAIClient openAIClient = new(
			new Uri(azureOpenAIEndpoint),
			new ApiKeyCredential(azureOpenAIKey));
ChatClient chatClient = openAIClient.GetChatClient(deploymentName);

// Extension methods to use data sources with options are subject to SDK surface changes. Suppress the
// warning to acknowledge and this and use the subject-to-change AddDataSource method.
//#pragma warning disable AOAI001

ChatCompletionOptions options = new();
options.AddDataSource(new AzureSearchChatDataSource()
{
	Endpoint = new Uri(searchEndpoint),
	IndexName = searchIndex,
	Authentication = DataSourceAuthentication.FromApiKey(searchKey),
});

ChatCompletion completion = chatClient.CompleteChat(
	[
		new UserChatMessage("What loads are available"),
			],
	options);

ChatMessageContext onYourDataContext = completion.GetMessageContext();

if (onYourDataContext?.Intent is not null)
{
	Console.WriteLine($"Intent: {onYourDataContext.Intent}");
}
foreach (ChatCitation citation in onYourDataContext?.Citations ?? [])
{
	Console.WriteLine($"Citation: {citation.Content}");
}

Error: (2,7): error CS0246: The type or namespace name 'Azure' could not be found (are you missing a using directive or an assembly reference?)
(3,14): error CS0234: The type or namespace name 'ClientModel' does not exist in the namespace 'System' (are you missing an assembly reference?)
(4,7): error CS0246: The type or namespace name 'Azure' could not be found (are you missing a using directive or an assembly reference?)
(5,7): error CS0246: The type or namespace name 'OpenAI' could not be found (are you missing a using directive or an assembly reference?)
(15,1): error CS0246: The type or namespace name 'AzureOpenAIClient' could not be found (are you missing a using directive or an assembly reference?)
(18,1): error CS0246: The type or namespace name 'ChatClient' could not be found (are you missing a using directive or an assembly reference?)
(24,1): error CS0246: The type or namespace name 'ChatCompletionOptions' could not be found (are you missing a using directive or an asse